## Notebook with code dedicated only to preprocess raw data

In [1]:
import sys
sys.path.append('../')
from source import Translate
from source import Process
from source import Align
import pandas as pd

/home/tiago/anaconda3/envs/databases/lib/python3.10/site-packages/Bio/Application/__init__.py:39: BiopythonDeprecationWarning: The Bio.Application modules and modules relying on it have been deprecated.

Due to the on going maintenance burden of keeping command line application
wrappers up to date, we have decided to deprecate and eventually remove these
modules.

We instead now recommend building your command line and invoking it directly
with the subprocess module.
  warnings.warn(


Insert tags to mark the source of each protein accordingly to its original database

In [2]:
date = "04052026"

In [3]:
Translate.ToProtein(
    input = f"../data/raw/megares.{date}.raw.fasta", 
    output = f"../data/filtered/megares.proteins.fasta", 
    minlength = 200)
Translate.ToProtein(
    input = f"../data/raw/resfinder.{date}.raw.fasta", 
    output = f"../data/filtered/resfinder.proteins.fasta", 
    minlength = 200
)

In [4]:
Process.InsertTag(
    input = f"../data/filtered/megares.proteins.fasta", 
    output = f"../data/filtered/megares.tagged.fasta", 
    tag = "MEGARES"
)
Process.InsertTag(
    input = f"../data/filtered/resfinder.proteins.fasta", 
    output = f"../data/filtered/resfinder.tagged.fasta", 
    tag = "RESFINDER"
)
Process.InsertTag(
    input = f"../data/raw/card.{date}.raw.fasta", 
    output = f"../data/filtered/card.tagged.fasta", 
    tag = "CARD"
)
Process.InsertTag(
    input = f"../data/raw/hmd.21032025.raw.fasta", 
    output = f"../data/filtered/hmd.tagged.fasta", 
    tag = "HMD"
)
Process.InsertTag(
    input = f"../data/raw/ncrd95.21032025.raw.fasta", 
    output = f"../data/filtered/ncrd95.tagged.fasta", 
    tag = "NCRD"
)
Process.InsertTag(
    input = f"../data/raw/ndaro.{date}.raw.fasta", 
    output = f"../data/filtered/ndaro.tagged.fasta", 
    tag = "NDARO"
)

In [5]:
!rm ../data/filtered/*.proteins.fasta

Concatenate all databases into one file

In [6]:
Process.ConcatenateFiles("../data/filtered/","../data/filtered/AllDatabases.tagged.fasta", "fasta")

Concatenate all FASTA ids into one file

In [7]:
Process.ConcatenateFiles("../data/filtered/","../data/filtered/AllDatabases.tagged.fasta_ids","fasta_ids")

Creates a diamond database

In [8]:
Process.CreateDatabase("../data/filtered/AllDatabases.tagged.fasta", "../data/filtered/AllDatabases.dmnd")

diamond v2.1.11.165 (C) Max Planck Society for the Advancement of Science, Benjamin Buchfink, University of Tuebingen
Documentation, support and updates available at http://www.diamondsearch.org
Please cite: http://dx.doi.org/10.1038/s41592-021-01101-x Nature Methods (2021)

#CPU threads: 12
Scoring parameters: (Matrix=BLOSUM62 Lambda=0.267 K=0.041 Penalties=11/1)
Database input file: ../data/filtered/AllDatabases.tagged.fasta
Opening the database file...  [0.049s]
Loading sequences...  [0.31s]
Masking sequences...  [0.203s]
Writing sequences...  [0.014s]
Hashing sequences...  [0.008s]
Loading sequences...  [0s]
Writing trailer...  [0s]
Closing the input file...  [0s]
Closing the database file...  [0.503s]

Database sequences  77149
  Database letters  25768626
     Database hash  c71600b9d14addca1ceb77b3029a3b2e
        Total time  1.091000s


In [9]:
Process.InsertTag(
    input = "../data/raw/uniprot.21032025.raw.fasta", 
    output = "../data/filtered/uniprot.tagged.fasta", 
    tag = "UNIPROT"
)

Run diamond to perform pairwise alignment. Essential to create sequence similarity networks

In [10]:
Align.RunDiamond(
    "../data/filtered/AllDatabases.tagged.fasta", 
    "../data/filtered/AllDatabases.dmnd", 
    "../data/filtered/AllDatabases.Paiwise.cov80.maxseq1.tsv",
    threads = 12,
    maxseq = 1,
    qcov = 80,
    minid = 0
)

Running: diamond blastp -d ../data/filtered/AllDatabases.dmnd -q ../data/filtered/AllDatabases.tagged.fasta -o ../data/filtered/AllDatabases.Paiwise.cov80.maxseq1.tsv -p 12 --id 0 --outfmt 6 qseqid sseqid stitle pident length qlen slen qstart qend sstart send evalue bitscore ppos full_qseq full_sseq -b 0.5 --query-cover 80 -k 1 --no-self-hits
diamond v2.1.11.165 (C) Max Planck Society for the Advancement of Science, Benjamin Buchfink, University of Tuebingen
Documentation, support and updates available at http://www.diamondsearch.org
Please cite: http://dx.doi.org/10.1038/s41592-021-01101-x Nature Methods (2021)

#CPU threads: 12
Scoring parameters: (Matrix=BLOSUM62 Lambda=0.267 K=0.041 Penalties=11/1)
Temporary directory: ../data/filtered
#Target sequences to report alignments for: 1
Opening the database...  [0.017s]
Database: ../data/filtered/AllDatabases.dmnd (type: Diamond database, sequences: 77149, letters: 25768626)
Block size = 500000000
Opening the input file...  [0.008s]
Open

In [ ]:
Align.RunDiamond(
    "../data/filtered/AllDatabases.tagged.fasta", 
    "../data/filtered/AllDatabases.dmnd", 
    "../data/filtered/AllDatabases.Paiwise.cov80.maxseq5.tsv",
    threads = 12,
    maxseq = 5,
    qcov = 80,
    minid = 0
)